In [56]:
import networkx as nx
import pandas as pd
import folium
from pyproj import Transformer
import matplotlib.pyplot as plt
import numpy as np
import branca

In [57]:
from google.colab import files
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()

Saving ChicagoSketch_net.tntp to ChicagoSketch_net (1).tntp


Saving ChicagoSketch_node.tntp to ChicagoSketch_node (1).tntp


Saving ChicagoSketch_trips.tntp to ChicagoSketch_trips (1).tntp


In [58]:

NET_FILE   = "ChicagoSketch_net.tntp"
NODE_FILE  = "ChicagoSketch_node.tntp"
TRIPS_FILE = "ChicagoSketch_trips.tntp"

# Converting to lat/lon
SOURCE_CRS = "EPSG:26771"
TARGET_CRS = "EPSG:4326"  # If you want lat/lon for mapping

###########################
# PARSE CHICAGO-SKETCH NET
###########################
def parse_chicago_sketch_net(net_file):
    """
    'free_flow_time' is the key weight for shortest-path analysis.
    Return: directed graph
    """
    G = nx.DiGraph()
    reading_data = False
    with open(net_file, "r") as f:
        for line in f:
            line = line.strip()

            if line.startswith("<END OF METADATA>"):
                reading_data = True
                continue

            if (not reading_data) or (not line) or line.startswith("~") or line.startswith("<"):
                continue

            # init_node, term_node, capacity, length, free_flow_time, b, power, speed, toll, link_type ;
            parts = line.split()
            if len(parts) >= 10:
                u = int(parts[0])
                v = int(parts[1])
                capacity = float(parts[2])
                length   = float(parts[3])
                fft      = float(parts[4])  # free_flow_time
                # b    = float(parts[5])
                # pwr  = float(parts[6])
                # spd  = float(parts[7])
                # toll = float(parts[8])
                # ltype= float(parts[9])

                # Add edge with free_flow_time as the key weight
                G.add_edge(u, v,
                           capacity=capacity,
                           length=length,
                           free_flow_time=fft)
    return G

###########################
# PARSE CHICAGO-SKETCH NODE FILE
###########################
from pyproj import Transformer

def parse_chicago_sketch_nodes(node_file, source_crs=None, target_crs=None):
    """
    Returns a DataFrame with columns: df_nodes = [Node, X, Y, Lon, Lat].
    """
    reading_data = False
    node_rows = []
    with open(node_file, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("node"):
                reading_data = True
                continue
            if (not reading_data) or (not line) or line.startswith("~") or line.startswith("<"):
                continue

            parts = line.split()
            # ex: "1  690309 1976022  ;"
            if len(parts) >= 3:
                nid = int(parts[0])
                x   = float(parts[1])
                y   = float(parts[2])
                node_rows.append((nid, x, y))

    df_nodes = pd.DataFrame(node_rows, columns=["Node", "X", "Y"])

    # We want lat/lon
    if source_crs and target_crs:
        transformer = Transformer.from_crs(source_crs, target_crs, always_xy=True)
        lons, lats = [], []
        for i, row in df_nodes.iterrows():
            X_ = row["X"]
            Y_ = row["Y"]
            lon, lat = transformer.transform(X_, Y_)
            lons.append(lon)
            lats.append(lat)
        df_nodes["Lon"] = lons
        df_nodes["Lat"] = lats

    return df_nodes

def attach_node_coords(G, df_nodes):
    """
    Attach node coords to the graph: G.nodes[n]["x"], ["y"], ["lat"], ["lon"]
    """
    for i, row in df_nodes.iterrows():
        n = row["Node"]
        if n in G:
            G.nodes[n]["x"] = row["X"]
            G.nodes[n]["y"] = row["Y"]
            if "Lat" in df_nodes.columns and "Lon" in df_nodes.columns:
                G.nodes[n]["lat"] = row["Lat"]
                G.nodes[n]["lon"] = row["Lon"]

###########################
# PARSE CHICAGO-SKETCH TRIPS
###########################
def parse_chicago_sketch_trips(trips_file):
    """
    Building an OD (origin/demand) dict: {(O, D): demand}.
    The format has lines like:
      Origin 1
        3 : 402.1 ;  5 : 25.66 ; ...
    """
    od_dict = {}
    reading_data = False
    current_origin = None
    with open(trips_file, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("<END OF METADATA>"):
                reading_data = True
                continue

            if (not reading_data) or (not line) or line.startswith("~") or line.startswith("<"):
                continue

            if line.startswith("Origin"):
                # ex "Origin 1"
                parts = line.split()
                current_origin = int(parts[1])
            else:
                # lines with " D : flow ; ..."
                parts = line.split(";")
                for part in parts:
                    part = part.strip()
                    if ":" in part:
                        dest_str, flow_str = part.split(":")
                        dest = int(dest_str)
                        flow = float(flow_str)
                        od_dict[(current_origin, dest)] = flow
    return od_dict


In [59]:
# Base Network Analysis

# Anikas Code to visualize the network
# Any Base Network Analysis that she did

In [60]:
# Hubs and Authorities Analysis:

# Parse the network
G = parse_chicago_sketch_net(NET_FILE)

# Compute edge betweenness centrality
edge_betweenness = nx.edge_betweenness_centrality(G, weight="free_flow_time")

# Convert to DataFrame for better visualization
df_edge_betweenness = pd.DataFrame(edge_betweenness.items(), columns=["Edge", "Betweenness Score"])
df_edge_betweenness = df_edge_betweenness.sort_values(by="Betweenness Score", ascending=False)


print(df_edge_betweenness.head())

print(np.mean(df_edge_betweenness["Betweenness Score"]))

# VISUALIZE ON MAP #

# Compute the center of the map based on node coordinates
df_nodes = parse_chicago_sketch_nodes(NODE_FILE, source_crs=SOURCE_CRS, target_crs=TARGET_CRS)
center_lat = df_nodes["Lat"].mean()
center_lon = df_nodes["Lon"].mean()

# Create a folium map centered on the network
m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

# Sort edges by betweenness centrality and take the top 30 for visualization
top_edges = df_edge_betweenness.head(50)

# Draw all roads in gray for context
for edge in G.edges(data=True):
    start, end = edge[:2]
    if start in df_nodes["Node"].values and end in df_nodes["Node"].values:
        lon1, lat1 = df_nodes.loc[df_nodes["Node"] == start, ["Lon", "Lat"]].values[0]
        lon2, lat2 = df_nodes.loc[df_nodes["Node"] == end, ["Lon", "Lat"]].values[0]
        folium.PolyLine([(lat1, lon1), (lat2, lon2)], color="gray", weight=2, opacity=0.2).add_to(m)

# Highlight top 30 edges by betweenness in red
for _, row in top_edges.iterrows():
    (start, end) = row["Edge"]
    if start in df_nodes["Node"].values and end in df_nodes["Node"].values:
        lon1, lat1 = df_nodes.loc[df_nodes["Node"] == start, ["Lon", "Lat"]].values[0]
        lon2, lat2 = df_nodes.loc[df_nodes["Node"] == end, ["Lon", "Lat"]].values[0]
        folium.PolyLine([(lat1, lon1), (lat2, lon2)], color="red", weight=3, opacity=0.8).add_to(m)

# Save and provide the map
m

            Edge  Betweenness Score
2571  (535, 486)           0.051592
2764  (486, 480)           0.050148
2743  (479, 478)           0.048250
2765  (486, 535)           0.047318
2731  (478, 477)           0.046298
0.003028081470710631


In [61]:
###########################
# BETWEENNESS WEIGHTED TSTT
###########################

def all_or_nothing_tstt(G, od_dict, weight="free_flow_time"):
    """
    For each OD pair, compute the shortest path cost (by 'weight') times the OD demand.
    Sum => total system travel time (TSTT).
    This is a simple/naive approach, ignoring capacity or congestion.
    This is our baseline metric for evaluating scenarios.
    """
    TSTT = 0.0
    for (O, D), demand in od_dict.items():
        if demand > 0 and G.has_node(O) and G.has_node(D):
            if nx.has_path(G, O, D):
                cost = nx.shortest_path_length(G, source=O, target=D, weight=weight)
                TSTT += cost * demand
    return TSTT

###########################
# IDENTIFY WORST OD PAIRS
###########################
def find_worst_od_pairs(G, od_dict, top_n=20, weight="free_flow_time"):
    """
    Return the top_n OD pairs by cost * demand, i.e. biggest total travel 'pain'.
    Each item is (O, D, cost, demand, cost*demand).
    """
    results = []
    for (O, D), dem in od_dict.items():
        if dem > 0 and G.has_node(O) and G.has_node(D) and nx.has_path(G, O, D):
            cost = nx.shortest_path_length(G, O, D, weight=weight)
            results.append((O, D, cost, dem, cost*dem))
    results.sort(key=lambda x: x[4], reverse=True)
    return results[:top_n]

###########################
# SCENARIO: PROPOSE NEW LINKS
###########################
def propose_new_links_based_on_worst_pairs(G, worst_pairs, num_links=5):
    """
    Very basic example: for top OD pairs, if there's no direct link, add one with half the current cost.
    Simulating a metro link, since our network consists of roads
    """
    new_edges = []
    for (O, D, cost, demand, cd) in worst_pairs:
        if not G.has_edge(O, D):
            # e.g. propose a link with half the cost
            new_edges.append((O, D, {"free_flow_time": cost*0.5}))
            if len(new_edges) >= num_links:
                break
    return new_edges

def apply_new_links(G, new_links):
    """
    Create a new scenario graph with these new links.
    """
    G_build = G.copy()
    for (u, v, attrs) in new_links:
        if not G_build.has_edge(u, v):
            G_build.add_edge(u, v, **attrs)
    return G_build

###########################
# MAIN
###########################

if __name__ == "__main__":
    # Load ChicagoSketch_net.tntp
    G = parse_chicago_sketch_net(NET_FILE)

    # Parse nodes & attach coordinates
    df_nodes = parse_chicago_sketch_nodes(NODE_FILE, source_crs=SOURCE_CRS, target_crs=TARGET_CRS)
    attach_node_coords(G, df_nodes)

    # Parse OD demand data
    od_dict = parse_chicago_sketch_trips(TRIPS_FILE)




    # Baseline Metric: this is before we make any changes to the network
    # baseline_tstt = all_or_nothing_tstt(G, od_dict, weight="free_flow_time")
    # print(f"Baseline TSTT: {baseline_tstt:.2f}")
    baseline_aspl = nx.average_shortest_path_length(G, weight="free_flow_time")

    # Identify worst OD pairs: the pairs that have the highest cost*demand value aka the biggest total travel 'pain'
    worst_pairs = find_worst_od_pairs(G, od_dict, top_n=20, weight="free_flow_time")
    print("Worst OD pairs (by cost*demand):")
    for wp in worst_pairs:
        print(wp)

    # Propose naive new links based on the worst OD pairs. This is simulating a metro link
    manual_link = (383, 387, {'free_flow_time': 50})
    new_links = propose_new_links_based_on_worst_pairs(G, worst_pairs, num_links=5)
    new_links.append(manual_link)
    print("Proposed new links:", new_links)

    # Build our new scenario, with the new links. "optimized" network

    G_build = apply_new_links(G, new_links)

    # Recompute Baseline on the new network
    build_aspl = nx.average_shortest_path_length(G_build, weight="free_flow_time")

    # build_tstt = all_or_nothing_tstt(G_build, od_dict, weight="free_flow_time")
    # print(f"Build TSTT: {build_tstt:.2f}")

    # # Compare. If travel time reduces, this means the scenario is beneficial
    delta_aspl = build_aspl - baseline_aspl
    print(f"Change in TSTT (build - baseline): {delta_aspl:.2f}")
    if delta_aspl < 0:
        print("Scenario improved total travel time in free-flow sense!")
    else:
        print("Scenario did not help or possibly made it worse in this simplistic model.")


Worst OD pairs (by cost*demand):
(377, 387, 103.97000000000003, 603.0, 62693.91000000002)
(387, 379, 110.03, 566.0, 62276.98)
(381, 387, 83.75, 722.0, 60467.5)
(357, 356, 10.23, 5042.63, 51586.104900000006)
(387, 381, 83.75, 572.0, 47905.0)
(387, 377, 103.97000000000001, 439.0, 45642.83000000001)
(387, 382, 101.5, 379.0, 38468.5)
(356, 357, 10.23, 2941.18, 30088.271399999998)
(379, 387, 110.03, 254.0, 27947.62)
(359, 356, 12.709999999999999, 1583.31, 20123.870099999996)
(383, 377, 130.17999999999998, 123.0, 16012.139999999998)
(358, 357, 8.64, 1694.31, 14638.8384)
(360, 357, 10.580000000000002, 1363.12, 14421.8096)
(357, 360, 10.580000000000002, 1336.37, 14138.794600000001)
(359, 357, 13.879999999999999, 855.08, 11868.5104)
(356, 36, 11.92, 959.75, 11440.22)
(29, 17, 14.33, 784.94, 11248.190200000001)
(382, 86, 75.64, 148.0, 11194.72)
(357, 358, 8.64, 1290.11, 11146.5504)
(382, 356, 87.77999999999999, 122.0, 10709.159999999998)
Proposed new links: [(377, 387, {'free_flow_time': 51.9850

Possible Interpretation of results:

What we did:
Identified the “biggest pain” OD pairs using cost × demand.
Added naive direct links (like a hypothetical metro line) for those pairs.
Shown that these new links reduce total network cost in a free-flow model.
Compared the resulting lines to Chicago’s actual rail system and found they align surprisingly well.

Why This Matters:
It demonstrates that network measures (like cost × demand) can guide where new transit lines might help the most.
Even a simple approach—ignoring real-world constraints like stops, capacity, or intermediate stations—points to corridors that real planners found essential.

Some Takeaways:
OD Matrix + Flow Data → Tells you where travelers come from and how costly it is to reach their destinations.
High Cost × Demand → A corridor with both high volume and high travel time is a prime candidate for transit improvements.
Naive “Direct Link” → Lowers the network’s total travel cost because it offers a fast shortcut for those big-demand corridors.
Framework Is Generalizable → The same logic applies to any city with an OD matrix and a road/transit network.
Real-World Validation → Observing that these naive direct links resemble actual Chicago lines confirms that data-driven network analysis can replicate or justify real transit decisions.

Possible Future Steps:
Refine the approach to include stops, capacity, or more realistic rail planning.
Incorporate dynamic or equilibrium assignment to account for congestion and mode choice.
Consider cost constraints, local politics, right-of-way, etc. for a full feasibility study.



In [62]:
# STILL WORKING ON CODE BELOW THIS:

def plot_worst_od_pairs(m, G, worst_pairs, title="Worst OD Pairs by cost*demand"):
    """
    Plots lines for the worst OD pairs on the given Folium map `m`.
    Colors them by cost*demand (the 'pain' metric).
    `worst_pairs` is a list of tuples: (O, D, cost, demand, cost*demand).
    """
    print(worst_pairs)
    # 1) Extract cost*demand values for color scale
    cd_values = [wp[4] for wp in worst_pairs]  # the 5th item is cost*demand
    if not cd_values:
        return  # no data
    min_cd, max_cd = min(cd_values), max(cd_values)
    if min_cd == max_cd:
        max_cd = min_cd + 1  # avoid zero-range colormap

    # 2) Create a colormap
    colormap = branca.colormap.LinearColormap(
        colors=["#7f0000", "#b30000", "#d7301f", "#ef6548", "#fc8d59"],  # Dark reds
        vmin=min_cd, vmax=max_cd
    )
    colormap.caption = title
    colormap.add_to(m)

    # # 3) Plot each OD pair as a PolyLine
    plotted_nodes = set()  # To avoid redundant markers
    for (O, D, cost, demand, cd) in worst_pairs:
        if "lat" in G.nodes[O] and "lat" in G.nodes[D]:
            latO, lonO = G.nodes[O]["lat"], G.nodes[O]["lon"]
            latD, lonD = G.nodes[D]["lat"], G.nodes[D]["lon"]
            color = colormap(cd)  # map cd => color
            popup_text = f"OD: {O}->{D}, Demand={demand}, Cost={cost:.2f}, c*d={cd:.2f}"
            folium.PolyLine(
                [(latO, lonO), (latD, lonD)],
                color=color,
                weight=3,
                opacity=0.7,
                popup=popup_text
            ).add_to(m)


            # Plot the Origin node
            if O not in plotted_nodes:
                folium.CircleMarker(
                    location=(latO, lonO),
                    radius=5,
                    color="blue",
                    fill=True,
                    fill_color="blue",
                    fill_opacity=0.8,
                    popup=f"Origin {O}"
                ).add_to(m)
                plotted_nodes.add(O)

            # Plot the Destination node
            if D not in plotted_nodes:
                folium.CircleMarker(
                    location=(latD, lonD),
                    radius=5,
                    color="red",
                    fill=True,
                    fill_color="red",
                    fill_opacity=0.8,
                    popup=f"Destination {D}"
                ).add_to(m)
                plotted_nodes.add(D)


def plot_new_links(m, G, new_links, color="blue", title="Proposed Links"):
    """
    Plots the newly proposed links on the Folium map.
    `new_links` is a list of (u, v, attrs).
    """
    for (u, v, attrs) in new_links:
        if u in G and v in G and "lat" in G.nodes[u] and "lat" in G.nodes[v]:
            latU, lonU = G.nodes[u]["lat"], G.nodes[u]["lon"]
            latV, lonV = G.nodes[v]["lat"], G.nodes[v]["lon"]
            popup_text = f"New link: {(latU, lonU)}->{(latV, lonV)}, free_flow_time={attrs.get('free_flow_time','?')}"
            folium.PolyLine(
                [(latU, lonU), (latV, lonV)],
                color=color,
                weight=4,
                opacity=0.9,
                popup=popup_text
            ).add_to(m)
    # adding between suburb link
    manual_latU, manual_lonU = (42.77824802099991, -87.917241135664)
    manual_latV, manual_lonV = (41.115687792096324, -88.51042429209119)
    folium.PolyLine([(manual_latU, manual_lonU), (manual_latV, manual_lonV)],color="blue",weight=4,opacity=0.9,popup=popup_text).add_to(m)



# Center the map on Chicago
# Or compute average lat/lon from df_nodes
m = folium.Map(location=[41.8781, -87.6298], zoom_start=9)
# Draw all roads in gray for context
for edge in G.edges(data=True):
    start, end = edge[:2]
    if start in df_nodes["Node"].values and end in df_nodes["Node"].values:
        lon1, lat1 = df_nodes.loc[df_nodes["Node"] == start, ["Lon", "Lat"]].values[0]
        lon2, lat2 = df_nodes.loc[df_nodes["Node"] == end, ["Lon", "Lat"]].values[0]
        folium.PolyLine([(lat1, lon1), (lat2, lon2)], color="gray", weight=2, opacity=0.2).add_to(m)

# Plot the worst OD pairs on the baseline graph
plot_worst_od_pairs(m, G_build, worst_pairs, title="Worst OD Pairs (Baseline)")


# # Plot the new links (on top)
plot_new_links(m, G_build, new_links, color="blue", title="Proposed New Links")

# Save & display
m


[(377, 387, 103.97000000000003, 603.0, 62693.91000000002), (387, 379, 110.03, 566.0, 62276.98), (381, 387, 83.75, 722.0, 60467.5), (357, 356, 10.23, 5042.63, 51586.104900000006), (387, 381, 83.75, 572.0, 47905.0), (387, 377, 103.97000000000001, 439.0, 45642.83000000001), (387, 382, 101.5, 379.0, 38468.5), (356, 357, 10.23, 2941.18, 30088.271399999998), (379, 387, 110.03, 254.0, 27947.62), (359, 356, 12.709999999999999, 1583.31, 20123.870099999996), (383, 377, 130.17999999999998, 123.0, 16012.139999999998), (358, 357, 8.64, 1694.31, 14638.8384), (360, 357, 10.580000000000002, 1363.12, 14421.8096), (357, 360, 10.580000000000002, 1336.37, 14138.794600000001), (359, 357, 13.879999999999999, 855.08, 11868.5104), (356, 36, 11.92, 959.75, 11440.22), (29, 17, 14.33, 784.94, 11248.190200000001), (382, 86, 75.64, 148.0, 11194.72), (357, 358, 8.64, 1290.11, 11146.5504), (382, 356, 87.77999999999999, 122.0, 10709.159999999998)]


In [63]:
# (-87.91729262080366 42.77831299719907)
m = folium.Map(location=[42.77831299719907, -87.91729262080366], zoom_start=12)
m

In [64]:
import folium
import random

def bucket_and_plot_od_custom(
    m,            # A Folium Map object
    G,            # Your NetworkX graph with lat/lon on each node
    od_dict       # Dict {(O, D): demand}
):
    """
    1. Use fixed bin edges: [0, 10, 100, 1000, 10000, 100000].
    2. Bucket OD pairs by demand into these bins.
    3. Randomly sample up to 'sample_size' OD pairs from each bin.
    4. Plot them on the Folium map, each bin in a distinct color.
    """

    # Custom bin edges
    bins = [1, 10, 100, 1000]
    n_bins = len(bins) - 1  # = 5 in this example

    sample_size = 15  # how many OD pairs to sample per bin

    # Prepare a list of lists to hold OD pairs for each bin
    bucket_od = [[] for _ in range(n_bins)]

    # Assign each OD pair to the appropriate bin
    for (O, D), demand in od_dict.items():
        for i in range(n_bins):
            if bins[i] <= demand < bins[i+1]:
                bucket_od[i].append((O, D, demand))
                break
        else:
            # If demand >= bins[-1], it doesn't fall in the < bins[i+1] check
            # So if demand is exactly 100000 or above, handle it:
            if demand >= bins[-1]:
                bucket_od[-1].append((O, D, demand))

    # Distinct colors for each bin
    color_list = ["blue", "green", "orange", "red", "purple", "brown", "gray"]

    for i in range(n_bins):
        c = color_list[i % len(color_list)]
        # Sample OD pairs in this bin
        if len(bucket_od[i]) > sample_size:
            chosen = random.sample(bucket_od[i], sample_size)
        else:
            chosen = bucket_od[i]

        # Plot each chosen OD pair
        for (O, D, dem) in chosen:
            if G.has_node(O) and G.has_node(D):
                if "lat" in G.nodes[O] and "lat" in G.nodes[D]:
                    latO, lonO = G.nodes[O]["lat"], G.nodes[O]["lon"]
                    latD, lonD = G.nodes[D]["lat"], G.nodes[D]["lon"]
                    popup_text = f"OD: {O}->{D}, Demand={dem:.2f}"
                    folium.PolyLine(
                        [(latO, lonO), (latD, lonD)],
                        color=c,
                        weight=3,
                        opacity=0.7,
                        popup=popup_text
                    ).add_to(m)

    print("Custom bins:", bins)
    print("OD pairs in each bin:", [len(b) for b in bucket_od])
    print("Plotted up to", sample_size, "OD pairs per bin with custom edges.")
# Assume you've parsed the ChicagoSketch data into G (with lat/lon) and od_dict
# For example:
# G = parse_chicago_sketch_net("ChicagoSketch_net.tntp")
# df_nodes = parse_chicago_sketch_nodes("ChicagoSketch_node.tntp", "EPSG:26771", "EPSG:4326")
# attach_node_coords(G, df_nodes)
# od_dict = parse_chicago_sketch_trips("ChicagoSketch_trips.tntp")

m = folium.Map(location=[41.8781, -87.6298], zoom_start=10)
m